In [1]:
import os 
import tempfile
from google.cloud import storage
import zipfile

c:\Users\fidyt\Desktop\Olist-ecommerce-project\.venv\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
from cryptography.fernet import Fernet

print(Fernet.generate_key().decode())

S0APJDcfC53drVnxtEJx8dNtvPRPT2q0Nd_DN3kin7A=


In [2]:
def download_kaggle_to_gcs(dataset_path,bucket_name,gcp_project_name):
    storage_client = storage.Client(gcp_project_name)
    bucket = storage_client.bucket(bucket_name)

    with tempfile.TemporaryDirectory() as tmp:
        command = f'kaggle datasets download -d {dataset_path} -p {tmp}'
        result =os.system(command)

        if result !=0:
            raise RuntimeError("Error with downloading kaggledataset")
        
        zip_file_name =[f for f in os.listdir(tmp) if f.endswith('.zip')][0]
        zip_file_path = os.path.join(tmp,zip_file_name)

        with zipfile.ZipFile(zip_file_path,'r') as zip:
            zip.extractall(tmp)

        for f in os.listdir(tmp):
            if f.endswith('.csv'):
                local_path = os.path.join(tmp,f)
                blob_path =f'raw_data/{f}'
                blob = bucket.blob(blob_path)

                print(f"Updating {f}")
                blob.upload_from_filename(local_path)
    print('All updated')

In [3]:
download_kaggle_to_gcs('olistbr/brazilian-ecommerce','olist_ecomerce_bucket','olist-ecomerce-project')

Updating olist_customers_dataset.csv
Updating olist_geolocation_dataset.csv
Updating olist_orders_dataset.csv
Updating olist_order_items_dataset.csv
Updating olist_order_payments_dataset.csv
Updating olist_order_reviews_dataset.csv
Updating olist_products_dataset.csv
Updating olist_sellers_dataset.csv
Updating product_category_name_translation.csv
All updated
